In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    StandardScaler,
    LabelEncoder
)
from pathlib import Path
from sklearn.impute import SimpleImputer

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "combined_shuffled.parquet"
)

In [2]:
df = pd.read_parquet(DATA_PATH)

---

In [3]:
df.duplicated().sum()

np.int64(3163)

In [4]:
df.isna().sum().sum()

np.int64(0)

In [5]:
df.columns

Index(['Src IP dec', 'Src Port', 'Dst IP dec', 'Dst Port', 'Protocol',
       'Timestamp', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets',
       'Total Length of Fwd Packet', 'Total Length of Bwd Packet',
       'Fwd Packet Length Max', 'Fwd Packet Length Min',
       'Fwd Packet Length Mean', 'Fwd Packet Length Std',
       'Bwd Packet Length Max', 'Bwd Packet Length Min',
       'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
       'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
       'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std',
       'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean',
       'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags',
       'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd RST Flags',
       'Bwd RST Flags', 'Fwd Header Length', 'Bwd Header Length',
       'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min',
       'Packet Length Max', 'Packet Length Mean', '

In [6]:
len(df['Label'].value_counts(dropna=False))

27

In [30]:
df['Label'].value_counts()

Label
BENIGN                                    1582539
Portscan                                   159066
DoS Hulk                                   158468
DDoS                                        95144
Infiltration - Portscan                     68626
DoS GoldenEye                                7567
Botnet - Attempted                           4067
FTP-Patator                                  3972
DoS Slowloris                                3859
DoS Slowhttptest - Attempted                 3368
SSH-Patator                                  2961
DoS Slowloris - Attempted                    1847
DoS Slowhttptest                             1740
Web Attack - Brute Force - Attempted         1292
Botnet                                        736
Web Attack - XSS - Attempted                  655
DoS Hulk - Attempted                          581
DoS GoldenEye - Attempted                      80
Web Attack - Brute Force                       73
Infiltration - Attempted                    

---

In [7]:
df.columns = df.columns.str.strip()

In [ ]:
df.drop_duplicates(inplace=True, ignore_index=True)

In [ ]:
drop_exact_cols = [
    'Fwd Segment Size Avg',
    'Bwd Segment Size Avg',
    'URG Flag Count',
    'Average Packet Size',
    'Attempted Category',
    'Src IP dec',
    'Src Port',
    'ICMP Code',
    'ICMP Type'
]

drop_calc_cols = [
    'Flow Packets/s',
    'Flow Bytes/s',
    'Fwd Packets/s',
    'Bwd Packets/s',
    'Down/Up Ratio'
]

drop_const_cols = [
    'Bwd URG Flags',
    'Fwd URG Flags'
]


droped = drop_exact_cols + drop_calc_cols + drop_const_cols
df.drop(columns=droped, inplace=True)

In [ ]:
df['Bin Label'] = np.where(df['Label'] == 'BENIGN', 'BENIGN', 'ATTACK')

Bin Label
BENIGN    1582539
ATTACK     514269
Name: count, dtype: int64

### **Neg. To NAN**

In [37]:
neg_cols = ['Fwd Header Length', 'Bwd Header Length']
for i, col in enumerate(neg_cols):
    mask = df[neg_cols[i]] < 0
    df[mask] = np.nan